<a href="https://colab.research.google.com/github/Ekimxii/fitness_pose_tracker/blob/karissa-processed-data/relable%20model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install the library

In [ ]:
!pip install -q transformers torch


Load the model

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Upload and read the CSV

In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

CSV_PATH = "qevd_feedback_text.csv"
df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

Saving qevd_feedback_text.csv to qevd_feedback_text (3).csv
Dataset shape: (2511, 5)
Columns: ['video_id', 'text', 'start_frame', 'end_frame', 'type']


,video_id,text,start_frame,end_frame,type
0,6,Nice!,461,502,feedback
1,6,Love the high knees!,503,567,feedback
2,6,I really like the speed here. Keep it up!,574,638,feedback
3,6,Awesome! Keep going!,664,727,feedback
4,6,We're almost there with the high knees. Work i...,781,843,feedback


In [ ]:
labels = [
    "correcting body posture or exercise form",
    "praising or motivating the person exercising",
    "explaining how to perform an exercise",
    "counting repetitions or reporting workout progress",
    "introducing or switching to another exercise",
    "telling the person to rest, stop, or take a break"
]

In [ ]:
label_map = {
    "correcting body posture or exercise form": "form_correction",
    "praising or motivating the person exercising": "encouragement",
    "explaining how to perform an exercise": "exercise_instruction",
    "counting repetitions or reporting workout progress": "rep_counting",
    "introducing or switching to another exercise": "transition",
    "telling the person to rest, stop, or take a break": "rest_safety"
}

labels = list(label_map.keys())

In [ ]:
import re
import pandas as pd

# Recognize digits and English number words (0–99).
ONES = (
    r"zero|one|two|three|four|five|six|seven|eight|nine|ten|"
    r"eleven|twelve|thirteen|fourteen|fifteen|sixteen|"
    r"seventeen|eighteen|nineteen"
)
TENS = r"twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety"
NUMBER = rf"(?:\d+|(?:{TENS})(?:[\s-](?:{ONES}))?|{ONES})"

def normalize_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower().replace("’", "'")

def counting_flags(text):
    text = normalize_text(text)

    # Examples: "10", "Five!", "one, two, three"
    pure_count = bool(re.fullmatch(
        rf"{NUMBER}(?:[\s,;.!-]+{NUMBER})*[\s.!?]*",
        text
    ))

    # Examples: "10 reps", "Three more!", "5 to go"
    explicit_count = bool(re.search(
        rf"\b{NUMBER}\s+(?:"
        rf"(?:more\s+)?(?:reps?|repetitions?)\b|"
        rf"more\b|to\s+go\b|left\b)",
        text
    ))

    # Do not treat time-only statements as rep counting.
    # "Rest for 10 seconds" therefore goes to the model.
    time_context = bool(re.search(
        r"\b(?:seconds?|minutes?|hours?)\b", text
    ))
    if time_context:
        explicit_count = bool(re.search(
            rf"\b{NUMBER}\s+(?:more\s+)?(?:reps?|repetitions?)\b",
            text
        ))

    return pure_count, explicit_count

def classify_feedback(text, original_type=None):
    original_type = normalize_text(original_type)

    # Highest priority: preserve original transition labels.
    if original_type in {"transition", "exercise_transition"}:
        return "transition", ["transition"]

    text = normalize_text(text)
    if not text:
        return "unknown", []

    pure_count, explicit_count = counting_flags(text)

    # Pure counts bypass BART completely.
    if pure_count:
        return "rep_counting", ["rep_counting"]

    # Other feedback keeps the existing model classification.
    result = classifier(
        text,
        candidate_labels=labels,
        multi_label=True
    )

    def category(label):
        mapped = label_map[label]
        return "transition" if mapped == "exercise_transition" else mapped

    primary = category(result["labels"][0])
    all_categories = [
        category(label)
        for label, score in zip(result["labels"], result["scores"])
        if score >= 0.6
    ]

    if primary not in all_categories:
        all_categories.insert(0, primary)

    # Keep other categories for mixed feedback, e.g.
    # "Great job! You've completed 10 reps!"
    if explicit_count:
        primary = "rep_counting"
        all_categories.insert(0, "rep_counting")

    return primary, list(dict.fromkeys(all_categories))

this is a test

In [ ]:
examples = [
    "Great job!",
    "Keep your back straight!",
    "Great job! Keep your back straight!",
    "Take a break if you feel tired.",
    "You've completed 10 reps!",
    "Let's move on to squats."
]

for text in examples:
    primary, all_labels = classify_feedback(text)

    print("Feedback:", text)
    print("Primary:", primary)
    print("All categories:", all_labels)
    print()

Feedback: Great job!
Primary: encouragement
All categories: ['encouragement']

Feedback: Keep your back straight!
Primary: form_correction
All categories: ['form_correction', 'exercise_instruction', 'encouragement']

Feedback: Great job! Keep your back straight!
Primary: encouragement
All categories: ['encouragement', 'form_correction', 'exercise_instruction']

Feedback: Take a break if you feel tired.
Primary: rest_safety
All categories: ['rest_safety', 'transition']

Feedback: You've completed 10 reps!
Primary: rep_counting
All categories: ['rep_counting', 'encouragement', 'exercise_instruction']

Feedback: Let's move on to squats.
Primary: transition
All categories: ['transition', 'exercise_instruction', 'encouragement', 'rep_counting', 'form_correction']



In [ ]:
print(df.columns.tolist())

['video_id', 'text', 'start_frame', 'end_frame', 'type']


In [ ]:
from tqdm.auto import tqdm

tqdm.pandas()

# Inspect the actual original labels.
print("Original type values:")
print(df["type"].value_counts(dropna=False))

original_transition = (
    df["type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["transition", "exercise_transition"])
)

# Pass BOTH text and original type to the hybrid classifier.
results = df.progress_apply(
    lambda row: classify_feedback(row["text"], row["type"]),
    axis=1
)

df["primary_category"] = [result[0] for result in results]
df["all_categories"] = [result[1] for result in results]

# Verify that every original transition was preserved.
assert df.loc[
    original_transition, "primary_category"
].eq("transition").all()

assert df.loc[
    original_transition, "all_categories"
].map(lambda categories: "transition" in categories).all()

print("\nOriginal transitions preserved:", original_transition.sum())
print("\nNew category distribution:")
print(df["primary_category"].value_counts())

display(
    df[["text", "type", "primary_category", "all_categories"]].head(20)
)

Original type values:
type
feedback      2013
transition     498
Name: count, dtype: int64


  0%|          | 0/2511 [00:00<?, ?it/s]


Original transitions preserved: 498

New category distribution:
primary_category
encouragement           999
transition              960
form_correction         249
rep_counting            139
rest_safety              95
exercise_instruction     69
Name: count, dtype: int64


,text,type,primary_category,all_categories
0,Nice!,feedback,transition,[transition]
1,Love the high knees!,feedback,encouragement,[encouragement]
2,I really like the speed here. Keep it up!,feedback,encouragement,[encouragement]
3,Awesome! Keep going!,feedback,encouragement,"[encouragement, rep_counting, transition]"
4,We're almost there with the high knees. Work i...,feedback,encouragement,"[encouragement, form_correction, exercise_inst..."
5,That was really good! keep it up for the next ...,feedback,encouragement,"[encouragement, rep_counting]"
6,That's a good one!,feedback,rest_safety,[rest_safety]
7,We are at 5 reps!,feedback,rep_counting,"[rep_counting, encouragement, exercise_instruc..."
8,10,feedback,rep_counting,[rep_counting]
9,15,feedback,rep_counting,[rep_counting]


In [ ]:
# Save the original data with the two new columns
df.to_csv("qevd_feedback_classified.csv", index=False)

# Download the updated CSV
from google.colab import files
files.download("qevd_feedback_classified.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>